# Colab T4 단백질 구조 설계와 재예측

Colab에서 런타임의 하드웨어 가속기를 **T4 GPU**로 선택한 뒤 위에서부터 셀을 실행합니다. 이 예제는 80잔기 단일 사슬 백본을 생성하고, 각 백본에 대해 리간드가 없는 조건에서 서열을 샘플링합니다. RF3 입력에는 MPNN 서열만 전달합니다. 재예측 결과와 백본의 일치도는 구조적 자기일관성을 나타냅니다.

## 기본값과 출처

| 단계 | 설정 |
| --- | --- |
| RFdiffusion3 | 길이 80, contig 없이 무조건 생성, 1배치 × 2백본, 200 diffusion steps, recycle은 체크포인트 기본값 |
| LigandMPNN | 백본당 10서열, batch size 10, 온도 0.1, seed 0 |
| RoseTTAFold3 | 서열당 diffusion 샘플 5개, 50 steps, 10 recycles, seed 0, 서열 단독 입력 |

길이, 생성 배치, MPNN 모델과 샘플 수는 [Foundry 공식 저장소 예제](https://github.com/RosettaCommons/foundry/blob/6866d610a9d5e485ef4bf601adb40c4928b8321f/examples/all.ipynb)의 값을 사용합니다. RFD3의 200 steps와 MPNN 온도는 공식 구현의 기본값입니다. RF3의 50 steps, 10 recycles, 샘플 5개는 예제가 호출하는 Python 엔진 기본값입니다. 생성된 두 백본의 모든 설계 서열을 재예측합니다. [RFD3 설정](https://github.com/RosettaCommons/foundry/blob/0d0b0ba44fab7ca76aa4b625c014ce5964c7d284/models/rfd3/configs/inference_engine/rfdiffusion3.yaml) · [RF3 엔진](https://github.com/RosettaCommons/foundry/blob/0d0b0ba44fab7ca76aa4b625c014ce5964c7d284/models/rf3/src/rf3/inference_engines/rf3.py)

## 설치와 유지보수 근거

- 2026-11-04 production 커밋은 문서 삭제였습니다. RFD3·RF3 실행 소스와 패키지 정의는 2026-10-16에 Foundry 1.0 패키지 구조로 바뀌었습니다. MPNN의 최근 실행 코드 수정은 같은 날 파서 옵션 반영 변경이었습니다. 공식 저장소 예제의 마지막 변경은 2025-12-17이었습니다. [production 커밋](https://github.com/RosettaCommons/foundry/commit/0d0b0ba44fab7ca76aa4b625c014ce5964c7d284) · [패키지 변경 PR](https://github.com/RosettaCommons/foundry/pull/404) · [MPNN 수정 PR](https://github.com/RosettaCommons/foundry/pull/389)
- 커뮤니티 Colab은 2026-10-20에 README와 실행 노트북을 갱신했습니다. SPDE 자료는 2026-10-06에 바인더 설계 노트북을 갱신했습니다. 실행 코드는 위 production 커밋으로 고정합니다. [커뮤니티 Colab 변경](https://github.com/amiralito/RFdiffusion3_Colab/commit/bc29cf6fd489fab92652e13ac06e14d51abefda2) · [SPDE 변경](https://github.com/pablo-arantes/spde-workshop-ai-protein-design/commit/60b802e95da403574dd52b1cd8f72034350b2c06)
- 고정 커밋의 Foundry는 Python 3.12 이상, PyTorch 2.11 이상, AtomWorks 3을 요구합니다. DGL과 구형 SE3Transformer를 설치하지 않으므로 해당 경로의 TLS, collections.Mapping, pooling import 오류를 피합니다. AtomWorks 3.0.2부터 바뀐 변환 생성자와의 충돌은 열린 이슈이며 여기서는 3.0.1로 고정합니다. [공식 의존성](https://github.com/RosettaCommons/foundry/blob/0d0b0ba44fab7ca76aa4b625c014ce5964c7d284/pyproject.toml) · [AtomWorks 이슈 #413](https://github.com/RosettaCommons/foundry/issues/413)
- T4에서 보고된 RF3 Triton 오류를 피하도록 cuEquivariance를 비활성화하고 RF3 attention을 vanilla로 지정합니다. RFD3는 low-memory와 PyTorch 커널을 사용하며 컴파일과 CUDA 그래프를 끕니다. 두 모델에 FP32를 적용합니다. 길이와 샘플 수는 위 표와 같습니다. 가중치는 공식 등록 항목에서 받습니다. [T4 이슈 #240](https://github.com/RosettaCommons/foundry/issues/240) · [Triton 이슈 #214](https://github.com/RosettaCommons/foundry/issues/214) · [PyTorch 공식 CUDA 휠](https://pytorch.org/get-started/previous-versions/) · [Colab 런타임 버전](https://research.google.com/colaboratory/runtime-version-faq.html)
- RFD3·RF3 CLI가 설치된 패키지에서 Hydra 설정 경로를 잘못 찾는 이슈와 수정 PR이 열려 있습니다. 아래 셀은 두 모델의 Python 추론 엔진을 직접 호출합니다. [Hydra 이슈 #403](https://github.com/RosettaCommons/foundry/issues/403) · [수정 PR #416](https://github.com/RosettaCommons/foundry/pull/416)
- 추론 성공과 VRAM 여유는 아래 Colab 셀에서 확인합니다. 모델별 프로세스 종료 시 GPU 메모리가 반환됩니다. 실패 시 셀에 오류 출력이 표시되고 런타임 폴더에 로그가 남습니다.

## 1. Colab 런타임 및 의존성 설정

커널의 Python·PyTorch·CUDA를 기록하고 Python 3.12 격리 환경에서 고정한 조합을 설치합니다. 설치 후 CUDA 연산, 의존성 일관성, 모델 모듈 import를 확인합니다.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.util import find_spec
import json
import os
import platform
import shutil
import subprocess
import sys
from uuid import uuid4

if find_spec("google.colab") is None:
    raise RuntimeError("Google Colab에서 실행하세요.")

commit = "0d0b0ba44fab7ca76aa4b625c014ce5964c7d284"
run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
root = Path("/content") / f"protein_design_{run_id}"
for directory in ("structures/generated", "structures/predicted", "structures/aligned",
                  "sequences", "metrics/rf3", "figures", "visualizations", "runtime", "config"):
    (root / directory).mkdir(parents=True, exist_ok=True)

work = Path("/content") / f"protein_work_{run_id}"
work.mkdir()
env_dir = Path("/content/foundry_0d0b0ba_py312")
python = env_dir / "bin/python"
weights = Path("/content/foundry_checkpoints")
weights.mkdir(exist_ok=True)
env = dict(os.environ, DESIGN_RUN=str(root), FOUNDRY_CHECKPOINT_DIRS=str(weights),
           DISABLE_CUEQUIVARIANCE="1", PYTHONUNBUFFERED="1")
env.pop("RFD3_LOW_MEMORY_MODE", None)

settings = {
    "commit": commit, "seed": 0,
    "rfd3": {"length": 80, "contig": None, "n_batches": 1, "diffusion_batch_size": 2,
             "num_timesteps": 200, "n_recycle": None, "low_memory_mode": True,
             "inference_kernel_backend": "torch", "compile_model": False,
             "compile_cuda_graphs": False, "precision": "32-true"},
    "mpnn": {"model_type": "ligand_mpnn", "batch_size": 10, "temperature": 0.1},
    "rf3": {"num_steps": 50, "n_recycles": 10, "diffusion_batch_size": 5,
            "triangle_attention_backend": "vanilla", "dense_attention_backend": "vanilla",
            "precision": "32-true", "msa": None, "template": None,
            "early_stopping_plddt_threshold": None},
}
(root / "config/settings.json").write_text(json.dumps(settings, indent=2))

def run_logged(args, name):
    args = list(map(str, args))
    with (root / "runtime" / f"{name}.log").open("a") as log:
        log.write("$ " + " ".join(args) + "\n")
        with subprocess.Popen(args, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            if process.wait():
                raise subprocess.CalledProcessError(process.returncode, args)

gpu = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name,memory.total,driver_version,compute_cap",
    "--format=csv,noheader"
], text=True).strip()
host_torch = subprocess.check_output([
    sys.executable, "-c", "import torch; print(torch.__version__, torch.version.cuda)"
], text=True).strip()
(root / "runtime/colab_host.json").write_text(json.dumps({
    "python": sys.version, "platform": platform.platform(), "torch_and_cuda": host_torch,
    "gpu_name_mib_driver_compute_cap": gpu,
}, indent=2))
print(sys.version, host_torch, gpu, sep="\n")
if "T4" not in gpu or not gpu.rstrip().endswith("7.5"):
    raise RuntimeError("Colab 런타임을 T4 GPU로 설정하세요.")

uv_command = [shutil.which("uv")] if shutil.which("uv") else []
if not uv_command:
    bootstrap = work / "uv_bootstrap"
    run_logged([sys.executable, "-m", "pip", "install", "--target", bootstrap, "uv"], "installation")
    uv_command = [bootstrap / "bin/uv"]

if not python.exists():
    run_logged([*uv_command, "venv", "--python", "3.12", env_dir], "installation")

torch_index = "https://download.pytorch.org/whl/cu128"
install = [*uv_command, "pip", "install", "--python", python]
constraint = work / "constraints.txt"
constraint.write_text("torch==2.11.0+cu128\natomworks==3.0.1\n")
run_logged([*install, "--index-url", torch_index, "torch==2.11.0+cu128"], "installation")
run_logged([
    *install, "--extra-index-url", torch_index, "--index-strategy", "unsafe-best-match",
    "--constraint", constraint, f"rc-foundry[rfd3] @ git+https://github.com/RosettaCommons/foundry.git@{commit}",
    "atomworks[ml]==3.0.1", "matplotlib", "py3Dmol",
], "installation")
run_logged([*uv_command, "pip", "check", "--python", python], "installation")
run_logged([*uv_command, "pip", "freeze", "--python", python], "packages")

(work / "pipeline_common.py").write_text(r'''
import csv
import json
import math
import os
from contextlib import contextmanager
from pathlib import Path
from time import perf_counter

import numpy as np
import torch

ROOT = Path(os.environ["DESIGN_RUN"])
CFG = json.loads((ROOT / "config/settings.json").read_text())
AA = dict(zip(
    "ALA ARG ASN ASP CYS GLU GLN GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL".split(),
    "ARNDCEQGHILKMFPSTWYV",
))

def json_ready(value):
    if isinstance(value, (np.generic, np.ndarray)):
        return json_ready(value.tolist())
    if isinstance(value, torch.Tensor):
        return json_ready(value.detach().cpu().tolist())
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

def write_json(path, value):
    path.write_text(json.dumps(json_ready(value), indent=2, ensure_ascii=False, allow_nan=False))

def write_csv(path, rows):
    if not rows:
        raise ValueError(f"No rows to write: {path}")
    columns = list(dict.fromkeys(key for row in rows for key in row))
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)

@contextmanager
def measure(model, operation, input_id):
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    baseline = torch.cuda.memory_allocated()
    start = perf_counter()
    status = "failed"
    try:
        yield
        torch.cuda.synchronize()
        status = "success"
    finally:
        row = dict(
            model=model, operation=operation, input_id=input_id, status=status,
            seconds=perf_counter() - start,
            baseline_allocated_mib=baseline / 2**20,
            peak_allocated_mib=torch.cuda.max_memory_allocated() / 2**20,
            peak_reserved_mib=torch.cuda.max_memory_reserved() / 2**20,
        )
        with (ROOT / "runtime/timings.jsonl").open("a") as handle:
            handle.write(json.dumps(row) + "\n")
        print(row, flush=True)

def save_cif(atoms, path):
    from atomworks.io.utils.io_utils import to_cif_file
    to_cif_file(atoms, path.with_suffix(""), file_type="cif")
    if not path.is_file():
        raise FileNotFoundError(path)

def load_cif(path):
    from biotite.structure.io import pdbx
    return pdbx.get_structure(pdbx.CIFFile.read(path), model=1)

def alpha_carbons(atoms):
    ca = atoms[(atoms.atom_name == "CA") & ~atoms.hetero]
    if not len(ca) or len(set(ca.chain_id)) != 1 or len(set(zip(ca.res_id, ca.ins_code))) != len(ca):
        raise ValueError("단일 사슬에서 잔기당 Cα 한 개가 필요합니다.")
    if not np.isfinite(ca.coord).all():
        raise ValueError("Cα에 유한하지 않은 좌표가 있습니다.")
    return ca

def sequence_of(atoms):
    return "".join(AA[str(name)] for name in alpha_carbons(atoms).res_name)

def show_structure(path, mobile=None):
    from tempfile import TemporaryDirectory
    from biotite.structure.io.pdb import PDBFile
    import py3Dmol

    def as_pdb(cif_path):
        with TemporaryDirectory() as tmp:
            target = Path(tmp) / "model.pdb"
            model = PDBFile()
            model.set_structure(load_cif(cif_path))
            model.write(target)
            return target.read_text()

    viewer = py3Dmol.view(width=780, height=480)
    viewer.addModel(as_pdb(path), "pdb")
    viewer.setStyle({"model": 0}, {"cartoon": {"color": "#2563eb"}})
    if mobile is not None:
        viewer.addModel(as_pdb(mobile), "pdb")
        viewer.setStyle({"model": 1}, {"cartoon": {"color": "#f97316"}})
    viewer.zoomTo()
    return viewer.write_html()
''', encoding="utf-8")

def run_stage(name, source):
    script = work / f"{name}.py"
    script.write_text(source, encoding="utf-8")
    run_logged([python, "-u", script], name)

In [ ]:
run_stage("verify_environment", r'''
import importlib.metadata as metadata
import json
import os
from pathlib import Path
import sys

import torch
import atomworks
from rfd3.engine import RFD3InferenceEngine
from mpnn.inference_engines.mpnn import MPNNInferenceEngine
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3._core import SHOULD_USE_CUEQUIVARIANCE
import pipeline_common

assert sys.version_info[:2] == (3, 12)
assert torch.__version__ == "2.11.0+cu128"
assert torch.version.cuda == "12.8"
assert torch.cuda.is_available() and torch.cuda.get_device_capability() == (7, 5)
assert "T4" in torch.cuda.get_device_name()
assert metadata.version("atomworks") == "3.0.1"
assert not SHOULD_USE_CUEQUIVARIANCE
assert torch.ones((2, 2), device="cuda").square().sum().item() == 4

dist = metadata.distribution("rc-foundry")
origin = json.loads(dist.read_text("direct_url.json") or "{}")
root = Path(os.environ["DESIGN_RUN"])
settings = json.loads((root / "config/settings.json").read_text())
assert origin["vcs_info"]["commit_id"] == settings["commit"]
(root / "runtime/environment.json").write_text(json.dumps({
    "python": sys.version, "torch": torch.__version__, "cuda_wheel": torch.version.cuda,
    "device": torch.cuda.get_device_name(), "compute_capability": torch.cuda.get_device_capability(),
    "vram_mib": torch.cuda.get_device_properties(0).total_memory / 2**20,
    "foundry": dist.version, "foundry_source": origin, "atomworks": metadata.version("atomworks"),
    "rf3_cuequivariance": SHOULD_USE_CUEQUIVARIANCE,
}, indent=2))
print((root / "runtime/environment.json").read_text())
''')

## 2. 공식 모델 가중치 가져오기

RFD3, LigandMPNN, RF3 체크포인트를 다운로드하고 파일명과 SHA-256을 저장합니다. 모델 초기화는 각 추론 단계에서 수행하고 시간을 별도로 측정합니다.

In [ ]:
run_logged([env_dir / "bin/foundry", "install", "rfd3", "ligandmpnn", "rf3",
            "--checkpoint-dir", weights], "checkpoints")

In [ ]:
run_stage("record_checkpoints", r'''
from hashlib import file_digest
from pathlib import Path
from pipeline_common import ROOT, write_json

from rfd3.checkpoints import REGISTERED_CHECKPOINTS as rfd3_weights
from mpnn.checkpoints import REGISTERED_CHECKPOINTS as mpnn_weights
from rf3.checkpoints import REGISTERED_CHECKPOINTS as rf3_weights

checkpoints = {}
for name, registry in [("rfd3", rfd3_weights), ("ligandmpnn", mpnn_weights), ("rf3", rf3_weights)]:
    entry = registry[name]
    path = Path(entry.get_default_path()).resolve()
    with path.open("rb") as handle:
        digest = file_digest(handle, "sha256").hexdigest()
    checkpoints[name] = {"path": str(path), "url": entry.url, "bytes": path.stat().st_size, "sha256": digest}
write_json(ROOT / "runtime/checkpoints.json", checkpoints)
''')

## 3. RFdiffusion3 백본 생성

입력은 80잔기 단일 사슬 설정입니다. 모델이 생성한 두 백본을 mmCIF로 저장하고 Cα 좌표와 잔기 수를 확인합니다.

In [ ]:
run_stage("generate_backbones", r'''
from pipeline_common import ROOT, CFG, alpha_carbons, measure, save_cif, write_json
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine

cfg = CFG["rfd3"]
with measure("RFD3", "load", "checkpoint"):
    options = RFD3InferenceConfig(
        specification={"length": cfg["length"], "extra": {}},
        diffusion_batch_size=cfg["diffusion_batch_size"],
        inference_sampler={"num_timesteps": cfg["num_timesteps"]},
        low_memory_mode=cfg["low_memory_mode"], compile_model=cfg["compile_model"],
        compile_cuda_graphs=cfg["compile_cuda_graphs"],
        inference_kernel_backend=cfg["inference_kernel_backend"], seed=CFG["seed"],
    )
    model = RFD3InferenceEngine(**options)
    model._assign_override("trainer.precision", cfg["precision"])
    resolved = model.initialize()
    assert str(resolved.trainer.precision) == cfg["precision"]

with measure("RFD3", "inference", "all_backbones"):
    output = model.run(inputs=None, n_batches=cfg["n_batches"], out_dir=None)
    samples = [sample for batch in output.values() for sample in batch]
    assert len(samples) == cfg["n_batches"] * cfg["diffusion_batch_size"]

manifest = []
for index, sample in enumerate(samples, 1):
    backbone_id = f"bb{index:02d}"
    relative = f"structures/generated/{backbone_id}.cif"
    assert len(alpha_carbons(sample.atom_array)) == cfg["length"]
    save_cif(sample.atom_array, ROOT / relative)
    manifest.append({"backbone_id": backbone_id, "file": relative, "example_id": sample.example_id})
    write_json(ROOT / f"config/{backbone_id}_rfd3_metadata.json", sample.metadata)
write_json(ROOT / "config/backbones.json", manifest)
''')

## 4. 생성 구조 시각화

백본별 py3Dmol HTML을 저장하고 첫 번째 백본을 Colab에 표시합니다. 저장한 HTML은 3Dmol.js를 인터넷에서 불러옵니다.

In [ ]:
run_stage("visualize_backbones", r'''
import json
from pipeline_common import ROOT, show_structure

for row in json.loads((ROOT / "config/backbones.json").read_text()):
    output = ROOT / "visualizations" / (row["backbone_id"] + ".html")
    output.write_text(show_structure(ROOT / row["file"]), encoding="utf-8")
''')

In [ ]:
from IPython.display import HTML, display
display(HTML(filename=str(root / "visualizations/bb01.html")))

## 5. LigandMPNN 서열 설계

각 RFD3 백본에서 10개 서열을 샘플링합니다. 설계 FASTA와 MPNN confidence·sequence recovery를 저장합니다. Sequence recovery는 MPNN 입력 백본에 기록된 잔기와의 일치도입니다.

In [ ]:
run_stage("design_sequences", r'''
import json

from mpnn.inference_engines.mpnn import MPNNInferenceEngine
from pipeline_common import ROOT, CFG, AA, measure, write_json

cfg = CFG["mpnn"]
with measure("LigandMPNN", "load", "checkpoint"):
    model = MPNNInferenceEngine(
        model_type=cfg["model_type"], is_legacy_weights=True, write_structures=False,
        write_fasta=False, out_directory=None, device="cuda",
    )

manifest = []
for row in json.loads((ROOT / "config/backbones.json").read_text()):
    backbone_id = row["backbone_id"]
    with measure("LigandMPNN", "inference", backbone_id):
        results = model.run(input_dicts=[{
            "structure_path": str(ROOT / row["file"]), "name": backbone_id,
            "batch_size": cfg["batch_size"], "temperature": cfg["temperature"],
            "seed": CFG["seed"], "remove_waters": True,
        }])
        assert len(results) == cfg["batch_size"]

    for index, result in enumerate(results, 1):
        sequence = result.output_dict["designed_sequence"]
        assert len(sequence) == CFG["rfd3"]["length"] and set(sequence) <= set(AA.values())
        manifest.append({
            "sequence_id": f"{backbone_id}_s{index:02d}", "backbone_id": backbone_id,
            "sequence": sequence, "reference": row["file"], "mpnn": result.output_dict,
        })
write_json(ROOT / "sequences/designs.json", manifest)
(ROOT / "sequences/designed.fasta").write_text("".join(
    f">{row['sequence_id']}\n{row['sequence']}\n" for row in manifest
))
''')

## 6. RF3 서열별 구조 재예측

설계 서열을 하나씩 RF3에 전달합니다. 모든 diffusion 샘플의 구조와 confidence 원본을 저장하고 RF3 ranking score로 서열별 대표 구조를 선택합니다.

T4의 FP32 실행에서는 RF3 Pairformer와 diffusion transformer가 입력을 BF16으로 강제 변환하는 설정을 끕니다. [Pairformer 구현](https://github.com/RosettaCommons/foundry/blob/0d0b0ba44fab7ca76aa4b625c014ce5964c7d284/models/rf3/src/rf3/model/layers/pairformer_layers.py#L221) · [Diffusion transformer 구현](https://github.com/RosettaCommons/foundry/blob/0d0b0ba44fab7ca76aa4b625c014ce5964c7d284/models/rf3/src/rf3/model/layers/af3_diffusion_transformer.py#L408)

In [ ]:
run_stage("refold_sequences", r'''
import json

from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.model.layers.af3_diffusion_transformer import AttentionPairBiasDiffusion
from rf3.model.layers.pairformer_layers import AttentionPairBiasPairformerDeepspeed
from rf3.utils.inference import InferenceInput
from pipeline_common import ROOT, CFG, alpha_carbons, measure, save_cif, sequence_of, write_json

cfg = CFG["rf3"]
with measure("RF3", "load", "checkpoint"):
    model = RF3InferenceEngine(
        ckpt_path="rf3", num_steps=cfg["num_steps"], n_recycles=cfg["n_recycles"],
        diffusion_batch_size=cfg["diffusion_batch_size"],
        triangle_attention_backend=cfg["triangle_attention_backend"],
        dense_attention_backend=cfg["dense_attention_backend"],
        early_stopping_plddt_threshold=cfg["early_stopping_plddt_threshold"],
        trainer_overrides={"precision": cfg["precision"]}, seed=CFG["seed"],
    )
    model.initialize()
    assert str(model.cfg.trainer.precision) == cfg["precision"]
    rf3_model = model.trainer.state["model"]
    for layer_type in (AttentionPairBiasPairformerDeepspeed, AttentionPairBiasDiffusion):
        layers = [layer for layer in rf3_model.modules() if isinstance(layer, layer_type)]
        if not layers or any(layer.use_deepspeed_evo for layer in layers):
            raise RuntimeError(f"Expected native RF3 attention layers for {layer_type.__name__}.")
        for layer in layers:
            layer.force_bfloat16 = False
        print(f"RF3 {layer_type.__name__}: disabled forced BF16 in {len(layers)} layers.")

manifest = []
for design in json.loads((ROOT / "sequences/designs.json").read_text()):
    sequence_id = design["sequence_id"]
    payload = {"name": sequence_id, "components": [{"chain_id": "A", "seq": design["sequence"]}]}
    write_json(ROOT / f"config/{sequence_id}_rf3_input.json", payload)
    with measure("RF3", "inference", sequence_id):
        input_spec = InferenceInput.from_json_dict(payload)
        predictions = model.run(inputs=input_spec, out_dir=None, annotate_b_factor_with_plddt=True)[sequence_id]
        assert len(predictions) == cfg["diffusion_batch_size"]

    ordered = sorted(predictions, key=lambda item: item.summary_confidences["ranking_score"], reverse=True)
    for rank, prediction in enumerate(ordered, 1):
        sample_id = f"{sequence_id}_r{rank:02d}"
        atoms = prediction.atom_array
        assert sequence_of(atoms) == design["sequence"]
        ca = alpha_carbons(atoms)
        relative = f"structures/predicted/{sample_id}.cif"
        save_cif(atoms, ROOT / relative)
        write_json(ROOT / f"metrics/rf3/{sample_id}.json", {
            "summary": prediction.summary_confidences, "confidence": prediction.confidences,
            "ca_plddt": ca.b_factor, "residue_ids": ca.res_id,
        })
        manifest.append({
            "sequence_id": sequence_id, "sample_id": sample_id, "sample_idx": prediction.sample_idx,
            "rank": rank, "file": relative, "reference": design["reference"],
            "backbone_id": design["backbone_id"],
        })
    write_json(ROOT / "runtime/predictions_progress.json", manifest)
write_json(ROOT / "config/predictions.json", manifest)
''')

## 7. 생성 구조와 재예측 구조 중첩

같은 설계의 백본과 RF3 구조를 잔기 순서대로 대응시킵니다. Kabsch 알고리즘으로 반사를 허용하지 않는 최적 회전과 평행이동을 구합니다. Cα-RMSD는 중첩한 Cα 원자의 제곱 거리 평균의 제곱근이며 단위는 Å입니다. 모든 RF3 샘플을 정렬한 mmCIF로 저장하고 대표 샘플은 구조 및 중첩 HTML로도 저장합니다.

In [ ]:
run_stage("compare_structures", r'''
import json

import numpy as np
from biotite.structure.io import pdbx
from pipeline_common import ROOT, alpha_carbons, load_cif, show_structure, write_csv, write_json

rows, confidence_rows = [], []
for prediction in json.loads((ROOT / "config/predictions.json").read_text()):
    sample_id = prediction["sample_id"]
    reference = ROOT / prediction["reference"]
    mobile_path = ROOT / prediction["file"]
    fixed_ca = alpha_carbons(load_cif(reference))
    mobile = load_cif(mobile_path)
    mobile_ca = alpha_carbons(mobile)
    if len(fixed_ca) != len(mobile_ca):
        raise ValueError(f"{sample_id}: unexpected residue count")
    for ca in (fixed_ca, mobile_ca):
        if not np.all(np.diff(ca.res_id) == 1):
            raise ValueError(f"{sample_id}: Cα residue indices are not contiguous")

    fixed_xyz, mobile_xyz = fixed_ca.coord.astype(float), mobile_ca.coord.astype(float)
    fixed_center, mobile_center = fixed_xyz.mean(axis=0), mobile_xyz.mean(axis=0)
    u, _, vt = np.linalg.svd((mobile_xyz - mobile_center).T @ (fixed_xyz - fixed_center))
    if np.linalg.det(u @ vt) < 0:
        u[:, -1] *= -1
    rotation = u @ vt
    delta = (mobile_xyz - mobile_center) @ rotation + fixed_center - fixed_xyz
    rmsd = float(np.sqrt(np.mean(np.sum(delta ** 2, axis=1))))

    aligned_path = ROOT / f"structures/aligned/{sample_id}.cif"
    cif = pdbx.CIFFile.read(mobile_path)
    atom_site = cif.block["atom_site"]
    axes = ("Cartn_x", "Cartn_y", "Cartn_z")
    xyz = np.column_stack([atom_site[key].as_array(float) for key in axes])
    aligned_xyz = (xyz - mobile_center) @ rotation + fixed_center
    for key, values in zip(axes, aligned_xyz.T):
        atom_site[key] = pdbx.CIFColumn(np.char.mod("%.4f", values))
    cif.write(aligned_path)
    if prediction["rank"] == 1:
        for label, html in [
            ("predicted", show_structure(mobile_path)),
            ("overlay", show_structure(reference, aligned_path)),
        ]:
            (ROOT / f"visualizations/{sample_id}_{label}.html").write_text(html, encoding="utf-8")

    data = json.loads((ROOT / f"metrics/rf3/{sample_id}.json").read_text())
    if data["residue_ids"] != mobile_ca.res_id.tolist() or len(data["ca_plddt"]) != len(mobile_ca):
        raise ValueError(f"{sample_id}: confidence and residue indices disagree")
    summary = {key: value for key, value in data["summary"].items()
               if value is None or isinstance(value, (bool, int, float))}
    rows.append(dict(prediction, ca_rmsd_angstrom=rmsd, generated_residues=len(fixed_ca),
                     predicted_residues=len(mobile_ca), **summary))
    for index, (res_id, plddt) in enumerate(zip(data["residue_ids"], data["ca_plddt"]), 1):
        confidence_rows.append({"sample_id": sample_id, "rank": prediction["rank"],
                                "position": index, "rf3_residue_id": res_id, "ca_plddt": plddt})

write_csv(ROOT / "metrics/structures.csv", rows)
write_json(ROOT / "metrics/structures.json", rows)
write_csv(ROOT / "metrics/residue_confidence.csv", confidence_rows)
write_json(ROOT / "metrics/residue_confidence.json", confidence_rows)
''')

In [ ]:
display(HTML(filename=str(root / "visualizations/bb01_s01_r01_predicted.html")))
display(HTML(filename=str(root / "visualizations/bb01_s01_r01_overlay.html")))

## 8. 서열 정렬, 정량 평가 및 그래프

모든 서열은 같은 길이의 무조건 생성 설계입니다. 정렬은 N말단부터 동일 위치를 대응시키며 gap을 넣지 않습니다. Sequence Identity는 전체 80개 위치 중 같은 잔기의 비율입니다. 다른 백본에서 나온 서열 간 비교도 이 위치 기준을 사용합니다.

pLDDT는 RF3가 반환한 전체 구조 요약과 각 잔기의 Cα 값을 저장합니다. pTM, PAE, PDE, 충돌 판정 등 RF3가 제공한 값은 원본 JSON에서 확인할 수 있습니다. 모델별 시간은 GPU 동기화를 경계로 측정한 경과 시간입니다. VRAM은 PyTorch 할당기 기준 최고 할당량과 예약량이며 CUDA 컨텍스트나 외부 라이브러리의 할당은 포함되지 않을 수 있습니다.

In [ ]:
run_stage("report_results", r'''
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from pipeline_common import ROOT, write_csv, write_json

designs = json.loads((ROOT / "sequences/designs.json").read_text())
samples = json.loads((ROOT / "metrics/structures.json").read_text())
confidence = json.loads((ROOT / "metrics/residue_confidence.json").read_text())
timings = [json.loads(line) for line in (ROOT / "runtime/timings.jsonl").read_text().splitlines()]
by_sequence = {row["sequence_id"]: row for row in samples if row["rank"] == 1}
ids = [row["sequence_id"] for row in designs]
representatives = [by_sequence[seq_id] for seq_id in ids]
write_csv(ROOT / "metrics/representative_structures.csv", representatives)
write_csv(ROOT / "metrics/mpnn.csv", [
    dict(sequence_id=row["sequence_id"], backbone_id=row["backbone_id"], **row["mpnn"])
    for row in designs
])

pairs = []
matrix = np.eye(len(designs))
alignment_text = []
first_by_backbone = {}
for i, row in enumerate(designs):
    sequence = row["sequence"]
    first = first_by_backbone.setdefault(row["backbone_id"], row)
    matches = "".join("|" if a == b else "." for a, b in zip(first["sequence"], sequence))
    alignment_text.append(
        f"{first['sequence_id']}  {first['sequence']}\n"
        f"{' ' * (len(first['sequence_id']) + 2)}{matches}\n"
        f"{row['sequence_id']}  {sequence}\n"
    )
    for j, other in enumerate(designs[:i]):
        fraction = sum(a == b for a, b in zip(sequence, other["sequence"])) / len(sequence)
        matrix[i, j] = matrix[j, i] = fraction
        pairs.append({"sequence_a": other["sequence_id"], "sequence_b": row["sequence_id"],
                      "positions": len(sequence), "identity_fraction": fraction})
(ROOT / "sequences/alignment.txt").write_text("\n".join(alignment_text))
(ROOT / "sequences/alignment.fasta").write_text((ROOT / "sequences/designed.fasta").read_text())
write_csv(ROOT / "metrics/sequence_identity.csv", pairs)
write_json(ROOT / "metrics/sequence_identity.json", pairs)

write_csv(ROOT / "runtime/timings.csv", timings)
write_json(ROOT / "runtime/timings.json", timings)
successful = [row for row in timings if row["status"] == "success" and row["operation"] == "inference"]
summary = []
for model in dict.fromkeys(row["model"] for row in successful):
    records = [row for row in successful if row["model"] == model]
    summary.append({"model": model, "calls": len(records),
                    "total_seconds": sum(row["seconds"] for row in records),
                    "peak_allocated_mib": max(row["peak_allocated_mib"] for row in records),
                    "peak_reserved_mib": max(row["peak_reserved_mib"] for row in records)})
write_csv(ROOT / "runtime/model_summary.csv", summary)
write_json(ROOT / "runtime/model_summary.json", summary)

def save(fig, filename):
    fig.savefig(ROOT / "figures" / filename, dpi=170, bbox_inches="tight")
    plt.close(fig)

def sequence_bars(key, label, filename, limits=None):
    fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
    ax.bar(ids, [row[key] for row in representatives], color="#2563eb")
    ax.tick_params(axis="x", labelrotation=65)
    ax.set_ylabel(label)
    if limits:
        ax.set_ylim(*limits)
    save(fig, filename)

sequence_bars("ca_rmsd_angstrom", "Cα-RMSD (Å)", "sequence_rmsd.png")
sequence_bars("overall_plddt", "RF3 overall pLDDT", "sequence_plddt.png")

confidence_by_sample = {}
for row in confidence:
    confidence_by_sample.setdefault(row["sample_id"], []).append(row["ca_plddt"])
fig, ax = plt.subplots(figsize=(11, 7), layout="constrained")
image = ax.imshow([confidence_by_sample[row["sample_id"]] for row in representatives],
                  aspect="auto", interpolation="nearest", cmap="viridis",
                  extent=(0.5, len(designs[0]["sequence"]) + 0.5, len(ids) - 0.5, -0.5))
ax.set_yticks(range(len(ids)), ids)
ax.set_xlabel("Residue position")
fig.colorbar(image, ax=ax, label="Cα pLDDT")
save(fig, "residue_confidence.png")

fig, ax = plt.subplots(figsize=(9, 8), layout="constrained")
image = ax.imshow(matrix, vmin=0, vmax=1, cmap="Blues")
ax.set_xticks(range(len(ids)), ids, rotation=75)
ax.set_yticks(range(len(ids)), ids)
fig.colorbar(image, ax=ax, label="Sequence identity fraction")
save(fig, "sequence_identity.png")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
names = [row["model"] for row in summary]
axes[0].bar(names, [row["total_seconds"] for row in summary], color="#2563eb")
axes[0].set_ylabel("Total inference time (s)")
positions = np.arange(len(names))
axes[1].bar(positions - 0.18, [row["peak_allocated_mib"] for row in summary],
            width=0.36, label="Allocated")
axes[1].bar(positions + 0.18, [row["peak_reserved_mib"] for row in summary],
            width=0.36, label="Reserved")
axes[1].set_xticks(positions, names)
axes[1].set_ylabel("Peak PyTorch VRAM (MiB)")
axes[1].legend()
save(fig, "inference_time_vram.png")

write_json(ROOT / "metrics/definitions.json", {
    "representative": "Highest RF3 ranking score per sequence.",
    "ca_rmsd_angstrom": "Kabsch superposition without reflection, C-alpha, matching single-chain residue order.",
    "overall_plddt": "RF3 overall_plddt, with the model's native scale.",
    "ca_plddt": "RF3 predicted pLDDT on each residue's C-alpha atom, with the model's native scale.",
    "identity_fraction": "Identical residues divided by full alignment length for equal-length, ungapped sequences.",
    "residue_count": "Count of distinct polymer residues with one finite C-alpha coordinate.",
    "time": "Wall clock elapsed with CUDA synchronization on each stage boundary.",
    "vram": "Per-process peak PyTorch allocator allocated and reserved memory, MiB.",
})
''')

In [ ]:
from IPython.display import Image
for filename in ("sequence_rmsd.png", "sequence_plddt.png",
                 "residue_confidence.png", "sequence_identity.png", "inference_time_vram.png"):
    display(Image(filename=str(root / "figures" / filename)))

## 9. 결과 ZIP 다운로드

완료된 실행의 구조, 서열, 평가 결과, 그래프, 3D HTML, 환경 로그, 설정을 하나로 묶습니다.

In [ ]:
from google.colab import files

if not (root / "metrics/definitions.json").is_file():
    raise RuntimeError("정량 평가 셀을 완료한 뒤 ZIP을 생성하세요.")
archive = shutil.make_archive(str(root), "zip", root_dir=root)
files.download(archive)